# NetSquid Ping-Pong Tutorial
in this notebook we will follow the first NetSquid tutorial : simulate a simple quantum game called Ping-Pong  

The game involves two players, Ping and Pong, who take turns measuring a **qubit** and sending it back to the other player through a quantum channel

This will help understand how quantum information can be transmitted between nodes in a network

## Concepts we'll see here

We will see the components, which model the physical parts of a quantum network:

- **Nodes**: Represent the players (Ping and Pong here)
- **Quantum channels**: Connect nodes and carry qubits
- **Connections**: Wrap channels for easier node communication

here, we create two nodes and connect them with two one-way quantum channels so the qubit can move in both directions.

In [ ]:
import netsquid as ns
qubits = ns.qubits.create_qubits(1)
qubits

in netsquid we use the measure() function to read the qubit's state and the create_qubits() function to create them.

In [20]:
qubit = qubits[0]
ns.qubits.reduced_dm(qubit)

array([[1.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j]])

In [6]:
ns.qubits.operate(qubit, ns.X)
ns.qubits.reduced_dm(qubit)

array([[0.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j]])

In [8]:
measurement_result, prob = ns.qubits.measure(qubit)
if measurement_result == 0:
    state = "|0>"
else:
    state = "|1>"
print(f"Measured {state} with probability {prob:.1f}")

Measured |1> with probability 1.0


In [9]:
measurement_result, prob = ns.qubits.measure(qubit, observable=ns.X)
if measurement_result == 0:
    state = "|+>"
else:
    state = "|->"
print(f"Measured {state} with probability {prob:.1f}")
ns.qubits.reduced_dm(qubit)

Measured |+> with probability 0.5


array([[0.5+0.j, 0.5+0.j],
       [0.5+0.j, 0.5+0.j]])

In [10]:
from netsquid.nodes import Node
node_ping = Node(name="Ping")
node_pong = Node(name="Pong")

The PingPongProtocol waits for an incoming qubit  
When a qubit arrives, it measures it in the node's preferred basis
The qubit is then sent back to the other node  
This process repeats until the simulation ends

Protocols let us describe how each node should act without worrying about low-level simulation details

In [11]:
from netsquid.components.models import DelayModel

class PingPongDelayModel(DelayModel):
    def __init__(self, speed_of_light_fraction=0.5, standard_deviation=0.05):
        super().__init__()
        # (the speed of light is about 300,000 km/s)
        self.properties["speed"] = speed_of_light_fraction * 3e5
        self.properties["std"] = standard_deviation
        self.required_properties = ['length']  # in km

    def generate_delay(self, **kwargs):
        avg_speed = self.properties["speed"]
        std = self.properties["std"]
        # The 'rng' property contains a random number generator
        # We can use that to generate a random speed
        speed = self.properties["rng"].normal(avg_speed, avg_speed * std)
        delay = 1e9 * kwargs['length'] / speed  # in nanoseconds
        return delay

In [12]:
from netsquid.components import QuantumChannel

distance = 2.74 / 1000  # default unit of length in channels is km
delay_model = PingPongDelayModel()
channel_1 = QuantumChannel(name="qchannel[ping to pong]",
                           length=distance,
                           models={"delay_model": delay_model})
channel_2 = QuantumChannel(name="qchannel[pong to ping]",
                           length=distance,
                           models={"delay_model": delay_model})

In [13]:
from netsquid.nodes import DirectConnection

connection = DirectConnection(name="conn[ping|pong]",
                              channel_AtoB=channel_1,
                              channel_BtoA=channel_2)
node_ping.connect_to(remote_node=node_pong, connection=connection,
                     local_port_name="qubitIO", remote_port_name="qubitIO")

('qubitIO', 'qubitIO')

In [14]:
from netsquid.protocols import NodeProtocol

class PingPongProtocol(NodeProtocol):
    def __init__(self, node, observable, qubit=None):
        super().__init__(node)
        self.observable = observable
        self.qubit = qubit
        # Define matching pair of strings for pretty printing of basis states:
        self.basis = ["|0>", "|1>"] if observable == ns.Z else ["|+>", "|->"]

    def run(self):
        if self.qubit is not None:
            # Send (TX) qubit to the other node via port's output:
            self.node.ports["qubitIO"].tx_output(self.qubit)
        while True:
            # Wait (yield) until input has arrived on our port:
            yield self.await_port_input(self.node.ports["qubitIO"])
            # Receive (RX) qubit on the port's input:
            message = self.node.ports["qubitIO"].rx_input()
            qubit = message.items[0]
            meas, prob = ns.qubits.measure(qubit, observable=self.observable)
            print(f"{ns.sim_time():5.1f}: {self.node.name} measured "
                  f"{self.basis[meas]} with probability {prob:.2f}")
            # Send (TX) qubit to the other node via connection:
            self.node.ports["qubitIO"].tx_output(qubit)

In [15]:
qubits = ns.qubits.create_qubits(1)
ping_protocol = PingPongProtocol(node_ping, observable=ns.Z, qubit=qubits[0])
pong_protocol = PingPongProtocol(node_pong, observable=ns.X)

to run:

Assign the PingPongProtocol to each node  
Start the protocols  
Run the simulation for a specific duration, here 300 ns  

The simulation prints out:

Which node measured the qubit  
The measurement outcome  
The probability of that outcome  

This shows the qubit being passed back and forth between the nodes

In [16]:
ping_protocol.start()
pong_protocol.start()
run_stats = ns.sim_run(duration=300)

 18.4: Pong measured |+> with probability 0.50
 37.1: Ping measured |0> with probability 0.50
 56.7: Pong measured |+> with probability 0.50
 74.0: Ping measured |1> with probability 0.50
 92.5: Pong measured |+> with probability 0.50
112.7: Ping measured |1> with probability 0.50
131.6: Pong measured |-> with probability 0.50
147.8: Ping measured |0> with probability 0.50
166.3: Pong measured |+> with probability 0.50
185.3: Ping measured |1> with probability 0.50
203.3: Pong measured |+> with probability 0.50
222.5: Ping measured |1> with probability 0.50
240.6: Pong measured |-> with probability 0.50
259.7: Ping measured |1> with probability 0.50
277.9: Pong measured |-> with probability 0.50
296.3: Ping measured |1> with probability 0.50


In [17]:
print(run_stats)


Simulation summary

Elapsed wallclock time: 0:00:00.020807
Elapsed simulation time: 3.00e+02 [ns]
Triggered events: 32
Handled callbacks: 32
Total quantum operations: 16
Frequent quantum operations: MEASURE = 16
Max qstate size: 1 qubits
Mean qstate size: 1.00 qubits

